# Reproduction with public dataset

This notebook reproduces the used methodology from the thesis with an public available dataset, to prove that the thesis is reproducable with any comparable dataset.

In [0]:
import kagglehub
import pandas as pd
import numpy as np
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
import joblib

In [0]:
# Download latest version
path = kagglehub.dataset_download("lucague/hospital-ehr-data-1171-patients-15-tables")

print("Path to dataset files:", path)

## Dataset creation

### Preprocessing

#### Load data and select necessary tables

In [0]:
dataframes = {}

# Find all CSV files in the downloaded dataset path
csv_files = glob.glob(os.path.join(path, "*.csv"))

# Loop through each file, load it, and store it in the dictionary
for file_path in csv_files:
    # Extract the table name (e.g., 'encounters' from 'encounters.csv')
    table_name = os.path.basename(file_path).replace('.csv', '')
    
    # Load the CSV and add it to the dictionary
    dataframes[table_name] = pd.read_csv(file_path)

    print(f"Table: {table_name}")
    display(dataframes[table_name].head(1))

In [0]:
# These 6 are necessary to combine, location, workforce capacity and production to one dataset
df_encounters = dataframes['encounters'].copy()
df_providers = dataframes['providers'].copy()
df_organizations = dataframes['organizations'].copy()
df_procedures = dataframes['procedures'].copy()
df_immunizations = dataframes['immunizations'].copy()

#### Create base dataset containing utilization, providers linked to the encounters

In [0]:
# Start with creating a base dataframe
df_encounters = dataframes['encounters'].copy()
df_organizations = dataframes['organizations'].copy()
df_providers = dataframes['providers'].copy()

# Rename ID columns
df_organizations = df_organizations.rename(columns={'Id': 'ORG_Id'})
df_providers = df_providers.rename(columns={'Id': 'PROV_Id'})

# Merge encounters and providers to add speciality
df_base = df_encounters.merge(
    df_providers[['PROV_Id', 'SPECIALITY']],
    left_on='PROVIDER',
    right_on='PROV_Id',
    how='left'
)

# Merge encounters and organizations to add utilization
df_base = df_base.merge(
    df_organizations[['ORG_Id', 'UTILIZATION']],
    left_on='ORGANIZATION',
    right_on='ORG_Id',
    how='left'
)

# Since base_cost en total_claim_cost are not accurate, we don't need those columns
columns_to_keep = [
    'Id', 'START', 'ORGANIZATION', 'PROVIDER', 
    'ENCOUNTERCLASS', 'SPECIALITY', 'UTILIZATION'
]

df_base = df_base[columns_to_keep]

#### Create billing_items dataset to calculate total revenue made for each category

In [0]:
# Add columns for procedure
df_procedures['CATEGORY'] = 'procedure'
df_procedures['DISPENSES'] = 1
df_procedures['TOTALCOST'] = df_procedures['BASE_COST']

df_encounters = df_encounters.rename(columns={'START': 'DATE', 'BASE_ENCOUNTER_COST': 'BASE_COST', 'Id': 'ENCOUNTER'})

# Add columns for immunization
df_immunizations['CATEGORY'] = 'immunization'
df_immunizations['DISPENSES'] = 1
df_immunizations['TOTALCOST'] = df_immunizations['BASE_COST']

# Add columns for encounters
df_encounters['CATEGORY'] = 'encounter'
df_encounters['DISPENSES'] = 1
df_encounters['TOTALCOST'] = df_encounters['BASE_COST']

cols = ['DATE', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'BASE_COST', 'DISPENSES', 'TOTALCOST', 'CATEGORY']

df_procedures = df_procedures[cols]
df_immunizations = df_immunizations[cols]
df_encounters = df_encounters[cols]

# Union 
df_billing_items = pd.concat([df_procedures, df_immunizations, df_encounters], ignore_index=True)

# Check if every unique code has the same price and description
df_code_check = df_billing_items.groupby(['CATEGORY', 'CODE', 'DESCRIPTION', 'DATE']).agg(
    unique_prices=('BASE_COST', 'nunique'),
    min_price=('BASE_COST', 'min'),
    max_price=('BASE_COST', 'max'),
).reset_index()

In [0]:
df_billing_items.head(5)

#### Check monetary values

In [0]:
df_code_check[df_code_check['unique_prices'] < 2]

Since the price does not seem to be adjusted over the years, we will not adjust for inflation

#### Join base dataset and billing items to one

In [0]:
df_encounters = dataframes['encounters'].copy()
df_organizations = dataframes['organizations'].copy()
df_providers = dataframes['providers'].copy()

# Rename ID columns
df_organizations = df_organizations.rename(columns={'Id': 'ORG_Id'})
df_providers = df_providers.rename(columns={'Id': 'PROV_Id'})

# Merge encounters and providers to add speciality
df_base = df_encounters.merge(
    df_providers[['PROV_Id', 'SPECIALITY']],
    left_on='PROVIDER',
    right_on='PROV_Id',
    how='left'
)

# Merge encounters and organizations to add utilization
df_base = df_base.merge(
    df_organizations[['ORG_Id', 'UTILIZATION']],
    left_on='ORGANIZATION',
    right_on='ORG_Id',
    how='left'
)

# Since base_cost en total_claim_cost are not accurate, we don't need those columns
columns_to_keep = [
    'Id', 'START', 'ORGANIZATION', 'PROVIDER', 
    'ENCOUNTERCLASS', 'SPECIALITY', 'UTILIZATION'
]

df_base = df_base[columns_to_keep]

In [0]:
df = df_base.merge(
    df_billing_items,
    left_on='Id',
    right_on='ENCOUNTER',
    how='right'
)

df = df.drop(columns=['ENCOUNTER'])

# ITEM_BASE_COST
df = df.rename(columns={'BASE_COST': 'ITEM_BASE_COST'})

# Bekijk het resultaat
display(df.head(10))

In [0]:
# Check if the join / union did not explode by summing the total cost for one specific organization
print(df[df['ORGANIZATION']=='ef58ea08-d883-3957-8300-150554edc8fb']['TOTALCOST'].sum())

org_id = 'ef58ea08-d883-3957-8300-150554edc8fb'
df_enc_org = dataframes['encounters'][dataframes['encounters']['ORGANIZATION'] == org_id]

enc_ids = df_enc_org['Id'].unique()

df_proc_org = dataframes['procedures'][dataframes['procedures']['ENCOUNTER'].isin(enc_ids)]
df_imm_org = dataframes['immunizations'][dataframes['immunizations']['ENCOUNTER'].isin(enc_ids)]

som_enc = df_enc_org['BASE_ENCOUNTER_COST'].sum()
som_proc = df_proc_org['BASE_COST'].sum()
som_imm = df_imm_org['BASE_COST'].sum()

print(som_enc + som_proc + som_imm)

#### Check missing values

In [0]:
df.isna().sum()

#### Check outliers per category

In [0]:
plt.figure(figsize=(12, 6))

sns.boxplot(data=df, x='CATEGORY', y='TOTALCOST')

plt.title('Spreiding van de Totale Kosten per Categorie')
plt.xlabel('Categorie')
plt.ylabel('Totale Kosten (€)')

plt.yscale('log') 

plt.show()

In [0]:
def get_outliers(group, column):
    Q1 = group[column].quantile(0.25)
    Q3 = group[column].quantile(0.75)
    IQR = Q3 - Q1
    
    ondergrens = Q1 - 1.5 * IQR
    bovengrens = Q3 + 1.5 * IQR
    
    return group[(group[column] < ondergrens) | (group[column] > bovengrens)]

df_outliers = df.groupby('CATEGORY').apply(lambda x: get_outliers(x, 'TOTALCOST')).reset_index(drop=True)
display(df_outliers.sort_values(by='TOTALCOST', ascending=False)[['DATE', 'CATEGORY', 'DESCRIPTION', 'TOTALCOST']].head(15))

We will keep the outliers, because treatments (procedures) can be very expensive in US

### Temporal Aggregation

In [0]:
df.head(5)

#### ML Dataset

In [0]:
df['DATUM'] = pd.to_datetime(df['DATE']).dt.date

df_grouped = df.groupby(['DATUM', 'ORGANIZATION', 'SPECIALITY', 'ENCOUNTERCLASS', 'CATEGORY' ]).agg(
    TOTALE_OMZET=('TOTALCOST', 'sum'),
    AANTAL_AFSPRAKEN=('Id', 'nunique'),
    UTILIZATION=('UTILIZATION', 'first')
).reset_index()

# Only based on date, organization and speciality so that it better compares to the full capacity of these specifications
df_artsen_capaciteit = df.groupby(['DATUM', 'ORGANIZATION', 'SPECIALITY']).agg(
    AANTAL_ARTSEN=('PROVIDER', 'nunique')
).reset_index()

df_timeseries = df_grouped.merge(
    df_artsen_capaciteit, 
    on=['DATUM', 'ORGANIZATION', 'SPECIALITY'], 
    how='left'
)

df_timeseries['DATUM'] = pd.to_datetime(df_timeseries['DATUM'])

df_timeseries.head(5)

In [0]:
print(df['TOTALCOST'].sum())
print(df_timeseries['TOTALE_OMZET'].sum())

In [0]:
# Filter on 2017-2019
df_timeseries = df_timeseries[(df_timeseries['DATUM'] >= '2017-01-01') & (df_timeseries['DATUM'] <= '2019-12-31')]

#### SARIMAX dataset

In [0]:
df_sarimax = df_timeseries.groupby('DATUM').agg(
    TOTALE_OMZET=('TOTALE_OMZET', 'sum')
).copy()
df_sarimax.sort_index(inplace=True)

### Feature Engineering

In [0]:
def add_calendar_features(df_raw, date_col='DATUM'):
    df = df_raw.copy()
    
    if date_col in df.columns:
        s_dates = pd.to_datetime(df[date_col].reset_index(drop=True))
    else:
        s_dates = pd.to_datetime(pd.Series(df.index))

    df['weekday']     = s_dates.dt.dayofweek.values
    df['month']       = s_dates.dt.month.values
    df['week']        = s_dates.dt.isocalendar().week.astype(int).values
    df['year']        = s_dates.dt.year.values
    df['quarter']     = s_dates.dt.quarter.values
    df['day_of_year'] = s_dates.dt.dayofyear.values

    df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
    df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)
    df['month_sin']   = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['month'] / 12)
    df['week_sin']    = np.sin(2 * np.pi * df['week'] / 52)
    df['week_cos']    = np.cos(2 * np.pi * df['week'] / 52)

    df['is_weekend']  = (df['weekday'] >= 5).astype(int)
    df['is_monday']   = (df['weekday'] == 0).astype(int)
    df['is_friday']   = (df['weekday'] == 4).astype(int)

    us_vacations = [
        ('winter',       '2016-12-22', '2017-01-03'), ('spring', '2017-03-11', '2017-03-26'), 
        ('summer',       '2017-06-01', '2017-08-15'), ('thanksgiving', '2017-11-22', '2017-11-26'),
        ('winter',       '2017-12-22', '2018-01-03'), ('spring', '2018-03-10', '2018-03-25'), 
        ('summer',       '2018-06-01', '2018-08-15'), ('thanksgiving', '2018-11-21', '2018-11-25'),
        ('winter',       '2018-12-21', '2019-01-03'), ('spring', '2019-03-09', '2019-03-24'), 
        ('summer',       '2019-06-01', '2019-08-15'), ('thanksgiving', '2019-11-27', '2019-12-01'),
        ('winter',       '2019-12-20', '2020-01-03'),
    ]
    
    for vtype in ['winter', 'spring', 'summer', 'thanksgiving']:
        df[f'is_{vtype}vakantie'] = 0
        
    for vtype, start, end in us_vacations:
        mask = (s_dates >= pd.to_datetime(start)) & (s_dates <= pd.to_datetime(end))
        df[f'is_{vtype}vakantie'] = np.where(mask.values, 1, df[f'is_{vtype}vakantie'])
        
    vakantie_cols = [f'is_{v}vakantie' for v in ['winter', 'spring', 'summer', 'thanksgiving']]
    df['is_schoolvakantie'] = (df[vakantie_cols].sum(axis=1) > 0).astype(int)

    us_federal_holidays = pd.to_datetime([
        '2017-01-02', '2017-01-16', '2017-02-20', '2017-05-29', '2017-07-04', '2017-09-04', '2017-10-09', '2017-11-10', '2017-11-23', '2017-12-25',
        '2018-01-01', '2018-01-15', '2018-02-19', '2018-05-28', '2018-07-04', '2018-09-03', '2018-10-08', '2018-11-12', '2018-11-22', '2018-12-25',
        '2019-01-01', '2019-01-21', '2019-02-18', '2019-05-27', '2019-07-04', '2019-09-02', '2019-10-14', '2019-11-11', '2019-11-28', '2019-12-25'
    ])
    
    df['is_feestdag'] = s_dates.isin(us_federal_holidays).astype(int).values
    df['is_dag_na_feestdag'] = s_dates.isin(us_federal_holidays + pd.Timedelta(days=1)).astype(int).values

    return df

    return df

#### SARIMAX feature engineering

In [0]:
df_sarimax = df_sarimax.asfreq('D').fillna(0)
df_sarimax = add_calendar_features(df_sarimax, date_col='DATUM')

#### ML feature engineering

In [0]:
df_timeseries = df_timeseries.reset_index(drop=False)
if 'index' in df_timeseries.columns:
    df_timeseries = df_timeseries.drop(columns=['index'])
df_calendar = add_calendar_features(df_timeseries)

In [0]:
cat_cols = [
    "ORGANIZATION",  
    "SPECIALITY", 
    "ENCOUNTERCLASS", 
    "CATEGORY", 
    "UTILIZATION",
]

df_ohe = pd.get_dummies(df_calendar, columns=cat_cols, drop_first=False)

bool_cols = df_ohe.select_dtypes(include=['bool', 'uint8']).columns
df_ohe[bool_cols] = df_ohe[bool_cols].astype(int)

df_model = df_ohe.sort_values('DATUM').copy()

## Model development

### SARIMAX

In [0]:
train = df_sarimax[df_sarimax['year'].isin([2017, 2018])]
test  = df_sarimax[df_sarimax['year'] == 2019]

print(f"Train: {train.index.min().date()} → {train.index.max().date()} ({len(train)} dagen)")
print(f"Test:  {test.index.min().date()} → {test.index.max().date()} ({len(test)} dagen)")

# Exo variables, same as used in the original model
exog_cols = [
    'is_feestdag',
    'is_dag_na_feestdag',
    'is_schoolvakantie',
]

# Train test split
y_train = train['TOTALE_OMZET']
y_test  = test['TOTALE_OMZET']
X_train = train[exog_cols]
X_test  = test[exog_cols]

# Rescale because Sarimax does not work with big values
scale_factor = 1000
y_train_scaled = y_train / scale_factor
y_test_scaled  = y_test  / scale_factor

# Start Grid Search
print("\nAIC Grid Search gestart (dit duurt ~5-10 minuten)...")

# Expanding window CV on trainset
tscv = TimeSeriesSplit(n_splits=2, gap=0)
results_grid = []

results_grid = []

for p in [0, 1]:
    for d in [0, 1]:
        for q in [0, 1]:
            for P in [0, 1]:
                for D in [0, 1]:
                    for Q in [0, 1]:
                        fold_maes = []
                        try:
                            for fold, (tr_idx, val_idx) in enumerate(tscv.split(y_train_scaled)):
                                y_tr  = y_train_scaled.iloc[tr_idx]
                                y_val = y_train_scaled.iloc[val_idx]
                                X_tr  = X_train.iloc[tr_idx]
                                X_val = X_train.iloc[val_idx]

                                model = SARIMAX(
                                    y_tr,
                                    exog=X_tr,
                                    order=(p, d, q),
                                    seasonal_order=(P, D, Q, 7),
                                    enforce_stationarity=False,
                                    enforce_invertibility=False
                                )
                                res = model.fit(disp=False, maxiter=200)
                                pred = res.forecast(steps=len(y_val), exog=X_val)
                                pred = pred.clip(lower=0)

                                mae = mean_absolute_error(y_val, pred)
                                fold_maes.append(mae)

                            results_grid.append({
                                'order': (p, d, q),
                                'seasonal': (P, D, Q, 7),
                                'mean_cv_MAE': round(np.mean(fold_maes) * scale_factor, 0),
                                'std_cv_MAE':  round(np.std(fold_maes)  * scale_factor, 0)
                            })
                            print(f"  order={(p,d,q)} seasonal={(P,D,Q,7)} → CV MAE: €{np.mean(fold_maes)*scale_factor:,.0f}")
                        except Exception as e:
                            continue

# Best model
results_df = pd.DataFrame(results_grid).sort_values('mean_cv_MAE')
print("\nTop 5:")
print(results_df.head(5).to_string(index=False))

best_order    = results_df.iloc[0]['order']
best_seasonal = results_df.iloc[0]['seasonal']

print(f"\nBeste specificatie: order={best_order}, seasonal={best_seasonal}")

# Train final model
print("\nFinaal model trainen...")

final_model = SARIMAX(
    y_train_scaled,
    exog=X_train,
    order=best_order,
    seasonal_order=best_seasonal,
    enforce_stationarity=False,
    enforce_invertibility=False
)
final_result = final_model.fit(disp=False)
print(final_result.summary())

# Predictions
test_pred_scaled = final_result.forecast(
    steps=len(test),
    exog=X_test
)

# Scale back
test_pred = test_pred_scaled * scale_factor 
test_pred = pd.Series(test_pred.values, index=y_test.index)
test_pred = test_pred.clip(lower=0)

In [0]:
pad_sarimax = './sarimax'
os.makedirs(pad_sarimax, exist_ok=True)

# Save model and split data in sarimax map
final_result.save(f'{pad_sarimax}/sarimax_public.pkl')

train.to_csv(f'{pad_sarimax}/train_public.csv', index=True)
test.to_csv(f'{pad_sarimax}/test_public.csv', index=True)
results_df.to_csv(f'{pad_sarimax}/results_df_public.csv', index=False)

X_train.to_csv(f'{pad_sarimax}/X_train_public.csv', index=True)
X_test.to_csv(f'{pad_sarimax}/X_test_public.csv', index=True)

pad_dataset = './dataset'
os.makedirs(pad_dataset, exist_ok=True)

# Save dataset used
df_sarimax.to_csv(f'{pad_dataset}/df_sarimax_public.csv', index=True)

In [0]:
test_evaluatie = test.copy()
test_evaluatie['Voorspelde_Omzet'] = test_pred.values

# Save test results from sarimax
test_evaluatie.to_csv(f'{pad_sarimax}/results_sarimax_public.csv', index=True)

# Also save grid_search
results_df.to_csv(f'{pad_sarimax}/grid_search_results_public.csv', index=False)

### XGBoost and RF

#### Train Test split

In [0]:
verboden_kolommen = ['DATUM', 'TOTALE_OMZET', 'time_index', 'AANTAL_AFSPRAKEN']
features = [col for col in df_model.columns if col not in verboden_kolommen]

df_model['year'] = df_model['year'].astype(int)
train_mask = df_model["year"].isin([2017, 2018])
test_mask = df_model["year"] == 2019

# Train test split
X_train, y_train = df_model.loc[train_mask, features], df_model.loc[train_mask, 'TOTALE_OMZET']
X_test, y_test = df_model.loc[test_mask, features], df_model.loc[test_mask, 'TOTALE_OMZET']

dates_train = df_model.loc[train_mask, 'DATUM']
dates_test = df_model.loc[test_mask, 'DATUM']

tscv = TimeSeriesSplit(n_splits=4)

In [0]:
pad_dataset = './dataset'
os.makedirs(pad_dataset, exist_ok=True)

# Save datasets
X_train.to_csv(f'{pad_dataset}/X_train_public.csv', index=False)
X_test.to_csv(f'{pad_dataset}/X_test_public.csv', index=False)
y_train.to_csv(f'{pad_dataset}/y_train_public.csv', index=False)
y_test.to_csv(f'{pad_dataset}/y_test_public.csv', index=False)
dates_train.to_csv(f'{pad_dataset}/dates_train_public.csv', index=False)
dates_test.to_csv(f'{pad_dataset}/dates_test_public.csv', index=False)

#### XGBoost training

In [0]:
# Model
param_dist = {
    'n_estimators': [100, 300, 500, 800],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 1.0, 5.0],
    'reg_lambda': [0.1, 1.0, 5.0, 10.0]
}

# base_model
xgb_base = xgb.XGBRegressor(random_state=42, n_jobs=-1)

# random search for hyperparameters
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20,                                 
    scoring='neg_mean_absolute_error',
    cv=tscv,
    verbose=1,                                 
    random_state=42,
    n_jobs=-1
)

# start model training
random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)

# feature importance
feature_importances = pd.DataFrame({
    'Feature': features,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\n--- Top 15 Belangrijkste Features ---")
print(feature_importances.head(15))

In [0]:
pad_xgb = './xgb_resultaten'
os.makedirs(pad_xgb, exist_ok=True)

# Save best model
best_model.save_model(f'{pad_xgb}/xgboost_public.json')

test_evaluatie = pd.DataFrame({
    'rapportagedatum': dates_test,
    'totale_omzet': y_test.values,
    'Voorspelde_Omzet': y_pred
})

# Save test results 
test_evaluatie.to_csv(f'{pad_xgb}/results_xgboost_public.csv', index=False)

#### RF Training

In [0]:
param_dist_rf = {
    'n_estimators': [100, 200],                     # Minder bomen per bos
    'max_depth': [10, 15],                          # Strenge limiet op de diepte
    'min_samples_split': [10, 20],               
    'min_samples_leaf': [4, 10],                 
    'max_features': ['sqrt'],                       # Alleen een sub-sample van features per boom
    'bootstrap': [True]                             
}

# base model
rf_base = RandomForestRegressor(random_state=42, n_jobs=2)

# random search
random_search_rf = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist_rf,
    n_iter=10,                                      # Minder iteraties (was 20)
    scoring='neg_mean_absolute_error',
    cv=tscv,  
    verbose=2,                                      # Verbose=2 laat je precies zien in de logs hoe ver hij is
    random_state=42,
    n_jobs=2                                        # Beperk het kopiëren van data in het RAM
)

# train model
random_search_rf.fit(X_train, y_train)
best_model_rf = random_search_rf.best_estimator_

# Predict on test set
y_pred_rf = best_model_rf.predict(X_test)

In [0]:
pad_rf = './rf_resultaten'
os.makedirs(pad_rf, exist_ok=True)

# Save best public model
joblib.dump(best_model_rf, f'{pad_rf}/rf_public.pkl')

# Create and save test results
test_evaluatie_rf = pd.DataFrame({
    'rapportagedatum': dates_test,
    'totale_omzet': y_test.values,
    'Voorspelde_Omzet': y_pred_rf
})

test_evaluatie_rf.to_csv(f'{pad_rf}/results_rf_public.csv', index=False)